# Conexión y validación
---

## Primer consulta: verificar conexión

In [0]:
SELECT current_database(), current_user, version();

## Segunda consulta: crear un esquema de trabajo

In [0]:
CREATE SCHEMA IF NOT EXISTS workshop;
SET search_path TO workshop;

# Modelar tablas (Schema-on-write)
---

In [0]:
CREATE TABLE workshop.locales_en_venta (
  direccion       VARCHAR(256),
  superficie_m2   INTEGER,
  preciousd       DOUBLE PRECISION,
  preciopeso      DOUBLE PRECISION,
  usdm2           DOUBLE PRECISION,
  pesosm2         DOUBLE PRECISION,
  antiguedad      INTEGER,
  en_galeria      VARCHAR(50),
  cotizacion      DOUBLE PRECISION,
  trimestre       VARCHAR(50),
  barrios         VARCHAR(128),
  comuna          INTEGER
)
DISTSTYLE AUTO
SORTKEY (barrios);

# Cargar con COPY
---

In [0]:
COPY workshop.locales_en_venta
FROM 's3://workshop-redshift-jip/curated/locales-en-venta/'
IAM_ROLE 'arn:aws:iam::955030484229:role/AWSRedshiftServerless-WorkshopRedshift'
FORMAT AS PARQUET;

## Validación del COPY

In [0]:
SELECT COUNT(*) AS total_filas FROM workshop.locales_en_venta;

SELECT * FROM workshop.locales_en_venta LIMIT 10;

# Consultas de negocio
---

## Primer consulta:

In [0]:
-- Top 10 barrios con mayor precio promedio por m2 (solo locales mayores a 20 m2)
SELECT 
  barrios,
  COUNT(*) AS cantidad_locales,
  TRUNC(AVG(usdm2), 2) AS precio_promedio_usdm2,
  AVG(superficie_m2) AS superficie_promedio_m2
FROM workshop.locales_en_venta
WHERE superficie_m2 > 20 AND usdm2 IS NOT NULL
GROUP BY barrios
ORDER BY precio_promedio_usdm2 DESC
LIMIT 10;

## Segunda consulta:

In [0]:
-- Cantidad total de locales y cuántos están en galería por comuna 
SELECT 
  comuna,
  COUNT(*) AS total_locales,
  SUM(CASE WHEN en_galeria = 'SI' THEN 1 ELSE 0 END) AS locales_en_galeria,
  SUM(preciousd) AS volumen_total_ofertado_usd
FROM workshop.locales_en_venta
WHERE preciousd > 0
GROUP BY comuna
ORDER BY comuna ASC;

## Tercer consulta:

In [0]:
-- Comparación del precio promedio por barrio frente al promedio general de su comuna
WITH PromedioComuna AS (
  SELECT 
    comuna, 
    TRUNC(AVG(usdm2),2) AS promedio_comuna_usdm2
  FROM workshop.locales_en_venta
  WHERE usdm2 IS NOT NULL
  GROUP BY comuna
)
SELECT 
  l.barrios,
  l.comuna,
  TRUNC(AVG(l.usdm2), 2) AS promedio_barrio_usdm2,
  pc.promedio_comuna_usdm2
FROM workshop.locales_en_venta l
JOIN PromedioComuna pc ON l.comuna = pc.comuna
WHERE l.usdm2 IS NOT NULL
GROUP BY l.barrios, l.comuna, pc.promedio_comuna_usdm2
ORDER BY promedio_barrio_usdm2 DESC
LIMIT 10;

# Cerrar el loop con el Data Lake (UNLOAD)
---

In [0]:
UNLOAD ('
  SELECT * FROM (
    SELECT 
      barrios,
      COUNT(*) AS cantidad_locales,
      TRUNC(AVG(usdm2), 2) AS precio_promedio_usdm2,
      AVG(superficie_m2) AS superficie_promedio_m2
    FROM workshop.locales_en_venta
    WHERE superficie_m2 > 20 AND usdm2 IS NOT NULL
    GROUP BY barrios
    ORDER BY precio_promedio_usdm2 DESC
    LIMIT 10
  )
')
TO 's3://workshop-redshift-jip/analytics/redshift/'
IAM_ROLE 'arn:aws:iam::955030484229:role/AWSRedshiftServerless-WorkshopRedshift'
FORMAT AS PARQUET
ALLOWOVERWRITE;